# Silver — Olist Marketing Funnel e IPCA mensal

Lê Bronze Delta; MQL filtradas de 01/06/2017 a 31/12/2017 pela data do primeiro contacto; Closed Deals ligados por `mql_id` sem filtrar `won_date` (preserva conversões posteriores); IPCA independente por mês. Grava apenas `silver.marketing_funnel` e `silver.ipca_mensal`. Não cria Gold.

In [1]:
DATA_INICIO = "2017-06-01"
DATA_FIM = "2017-12-31"

In [2]:
from pyspark.sql import SparkSession, functions as F
from delta import configure_spark_with_delta_pip
HDFS = "hdfs://hdfs-nn:9000"
WAREHOUSE = f"{HDFS}/warehouse"
builder = (SparkSession.builder.appName("EDSTD Bruno Marketing")
    .config("spark.sql.warehouse.dir", WAREHOUSE)
    .config("hive.metastore.uris", "thrift://hive-metastore:9083")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.jars.packages", "io.delta:delta-core_2.12:2.4.0")
    .enableHiveSupport())
spark = configure_spark_with_delta_pip(builder).getOrCreate()
spark.sparkContext.setLogLevel("WARN")
print("Spark:", spark.version)

Spark: 3.4.1


In [3]:
spark.sql(f"CREATE DATABASE IF NOT EXISTS silver LOCATION '{WAREHOUSE}/silver.db/'")
mql_raw = spark.table("bronze.bronze_mql")
deals_raw = spark.table("bronze.bronze_closed_deals")
ipca_raw = spark.table("bronze.bronze_ipca")
for nome, df in (("MQL", mql_raw), ("Closed Deals", deals_raw), ("IPCA", ipca_raw)):
    print(nome, df.count(), "linhas")
    df.printSchema()

MQL 8000 linhas
root
 |-- mql_id: string (nullable = true)
 |-- first_contact_date: string (nullable = true)
 |-- landing_page_id: string (nullable = true)
 |-- origin: string (nullable = true)
 |-- _ficheiro_origem: string (nullable = true)
 |-- _data_ingestao: timestamp (nullable = true)

Closed Deals 842 linhas
root
 |-- mql_id: string (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- sdr_id: string (nullable = true)
 |-- sr_id: string (nullable = true)
 |-- won_date: string (nullable = true)
 |-- business_segment: string (nullable = true)
 |-- lead_type: string (nullable = true)
 |-- lead_behaviour_profile: string (nullable = true)
 |-- has_company: string (nullable = true)
 |-- has_gtin: string (nullable = true)
 |-- average_stock: string (nullable = true)
 |-- business_type: string (nullable = true)
 |-- declared_product_catalog_size: string (nullable = true)
 |-- declared_monthly_revenue: string (nullable = true)
 |-- _ficheiro_origem: string (nullable = true)
 |--

## 1. Limpeza e filtro MQL

In [4]:
mql = (mql_raw
    .withColumn("mql_id", F.trim("mql_id"))
    .withColumn("first_contact_date", F.to_date("first_contact_date", "yyyy-MM-dd"))
    # origem em falta ou "unknown" fica NULL na Silver (na Gold passa ao registo -1 da dimensão)
    .withColumn("origin", F.when(F.col("origin").isNull() | F.lower(F.trim("origin")).isin("", "unknown"), F.lit(None).cast("string"))
                .otherwise(F.lower(F.trim("origin"))))
    .withColumn("contact_month", F.trunc("first_contact_date", "month"))
    .filter(F.col("first_contact_date").between(DATA_INICIO, DATA_FIM))
    .select("mql_id", "first_contact_date", "contact_month", "landing_page_id", "origin"))
assert mql.filter(F.col("mql_id").isNull() | (F.col("mql_id") == "")).count() == 0
assert mql.groupBy("mql_id").count().filter("count > 1").count() == 0
print("MQL no período:", mql.count())

MQL no período: 2002


## 2. Closed Deals — Boolean e tipos

In [5]:
def para_booleano(coluna):
    v = F.lower(F.trim(F.col(coluna)))
    return (F.when(v.isin("true", "1", "yes", "y", "sim", "s"), F.lit(True))
             .when(v.isin("false", "0", "no", "n", "não", "nao"), F.lit(False))
             .otherwise(F.lit(None).cast("boolean")))
for coluna in ("has_company", "has_gtin"):
    print("Valores originais:", coluna)
    deals_raw.groupBy(coluna).count().orderBy(F.desc("count")).show(30, truncate=False)
    v = F.lower(F.trim(F.col(coluna)))
    invalidos = deals_raw.filter(F.col(coluna).isNotNull() & (v != "") &
       ~v.isin("true", "false", "1", "0", "yes", "no", "y", "n", "sim", "s", "não", "nao"))
    assert invalidos.count() == 0, f"Valor booleano inesperado em {coluna}"
deals = (deals_raw
    .withColumn("mql_id", F.trim("mql_id"))
    .withColumn("seller_id", F.trim("seller_id"))
    .withColumn("won_date", F.to_timestamp("won_date", "yyyy-MM-dd HH:mm:ss"))
    .withColumn("declared_monthly_revenue", F.col("declared_monthly_revenue").cast("double"))
    .withColumn("declared_product_catalog_size", F.col("declared_product_catalog_size").cast("integer"))
    .withColumn("has_company", para_booleano("has_company"))
    .withColumn("has_gtin", para_booleano("has_gtin"))
    .withColumn("average_stock", F.when(F.col("average_stock").isNull() |
         (F.trim("average_stock") == "") | (F.lower(F.trim("average_stock")) == "unknown"),
         F.lit(None).cast("string")).otherwise(F.trim("average_stock")))
    .drop("_data_ingestao", "_ficheiro_origem"))
assert deals.filter(F.col("mql_id").isNull() | (F.col("mql_id") == "")).count() == 0
assert deals.groupBy("mql_id").count().filter("count > 1").count() == 0
print("Closed Deals:", deals.count())

Valores originais: has_company
+-----------+-----+
|has_company|count|
+-----------+-----+
|null       |779  |
|True       |58   |
|False      |5    |
+-----------+-----+

Valores originais: has_gtin
+--------+-----+
|has_gtin|count|
+--------+-----+
|null    |778  |
|True    |54   |
|False   |10   |
+--------+-----+

Closed Deals: 842


## 3. Integração das leads

In [6]:
marketing = (mql.join(deals.withColumn("converted", F.lit(1)), "mql_id", "left")
    .withColumn("converted", F.coalesce(F.col("converted"), F.lit(0))))
assert marketing.count() == mql.count(), "JOIN alterou número de leads"
assert marketing.groupBy("mql_id").count().filter("count > 1").count() == 0
print("Leads:", marketing.count())
print("Convertidas:", marketing.filter("converted = 1").count())
marketing.select("mql_id", "first_contact_date", "origin", "seller_id", "converted").show(5, truncate=False)

Leads: 2002
Convertidas: 61
+--------------------------------+------------------+--------------+---------+---------+
|mql_id                          |first_contact_date|origin        |seller_id|converted|
+--------------------------------+------------------+--------------+---------+---------+
|8c18d1de7f67e60dbd64e3c07d7e9d5d|2017-10-20        |paid_search   |null     |0        |
|2f838cade4a6012a6cb1016d1d8d95ed|2017-11-10        |organic_search|null     |0        |
|7281942387a1a0c3f72a50a8b0bb0920|2017-12-25        |social        |null     |0        |
|a5448d7164426cc7f3260a7b2001e3e6|2017-12-08        |paid_search   |null     |0        |
|443a4c031f05c42f729a1ff2ac219b5a|2017-07-25        |paid_search   |null     |0        |
+--------------------------------+------------------+--------------+---------+---------+
only showing top 5 rows



## 4. IPCA mensal — dimensão futura

In [7]:
ipca = (ipca_raw
    .withColumn("reference_month", F.to_date(F.concat(F.lit("01/"), F.col("Date")), "dd/MM/yyyy"))
    .withColumn("ipca_rate_12m", F.col("IPCA Rate (last 12 months)").cast("double"))
    .select("reference_month", "ipca_rate_12m")
    .filter(F.col("reference_month").between(DATA_INICIO, DATA_FIM)))
assert ipca.filter(F.col("reference_month").isNull() | F.col("ipca_rate_12m").isNull()).count() == 0
assert ipca.groupBy("reference_month").count().filter("count > 1").count() == 0
assert ipca.count() == 7, "Esperados sete meses entre junho e dezembro de 2017"
ipca.orderBy("reference_month").show(truncate=False)

+---------------+-------------+
|reference_month|ipca_rate_12m|
+---------------+-------------+
|2017-06-01     |3.0          |
|2017-07-01     |2.71         |
|2017-08-01     |2.46         |
|2017-09-01     |2.54         |
|2017-10-01     |2.7          |
|2017-11-01     |2.8          |
|2017-12-01     |2.95         |
+---------------+-------------+



## 5. Escrita Delta e validação

In [8]:
def gravar_silver(df, tabela):
    (df.write.format("delta").mode("overwrite").option("overwriteSchema", "true")
       .option("path", f"{WAREHOUSE}/silver.db/{tabela}/")
       .saveAsTable(f"silver.{tabela}"))
    print(f"silver.{tabela}: {spark.table(f'silver.{tabela}').count()} linhas")
gravar_silver(marketing, "marketing_funnel")
gravar_silver(ipca, "ipca_mensal")

silver.marketing_funnel: 2002 linhas
silver.ipca_mensal: 7 linhas


In [9]:
mf = spark.table("silver.marketing_funnel")
im = spark.table("silver.ipca_mensal")
assert mf.count() == mql.count()
assert mf.select("mql_id").distinct().count() == mf.count()
assert im.count() == 7
assert mf.schema["has_company"].dataType.simpleString() == "boolean"
assert mf.schema["has_gtin"].dataType.simpleString() == "boolean"
print("Convertidas sem seller_id:", mf.filter("converted = 1 AND seller_id IS NULL").count())
mf.agg(F.count("*").alias("leads"), F.sum("converted").alias("convertidas"),
    F.round(F.avg(F.col("converted").cast("double")) * 100, 2).alias("taxa_conversao_pct")).show()
spark.sql("SHOW TABLES IN silver").show(truncate=False)
for t in ("marketing_funnel", "ipca_mensal"):
    spark.sql(f"DESCRIBE DETAIL silver.{t}").select("format", "location", "numFiles").show(truncate=False)

Convertidas sem seller_id: 0
+-----+-----------+------------------+
|leads|convertidas|taxa_conversao_pct|
+-----+-----------+------------------+
| 2002|         61|              3.05|
+-----+-----------+------------------+

+---------+-------------------------+-----------+
|namespace|tableName                |isTemporary|
+---------+-------------------------+-----------+
|silver   |amazon_avaliacoes        |false      |
|silver   |amazon_produtos          |false      |
|silver   |avaliacoes               |false      |
|silver   |avaliacoes_rtr           |false      |
|silver   |dataco_itens             |false      |
|silver   |datafiniti_amazon_ligacao|false      |
|silver   |datafiniti_precos        |false      |
|silver   |datafiniti_produtos      |false      |
|silver   |feriados                 |false      |
|silver   |inmet_meteorologia_diaria|false      |
|silver   |ipca_mensal              |false      |
|silver   |marketing_funnel         |false      |
|silver   |olist_avaliaco

In [10]:
spark.stop()